# Gasoil COT Mapping: ICE Europe → Marouen's cot_db

Goal: reconstruct the QS rows in `cot_db.csv` from ICE Europe data.

Same pattern as Brent — ICE-Europe-only, no CFTC legacy report available.
- Commercial = Producer/Merchant (Combined)
- ManagedMoney = Managed Money + Other Reportables (Combined)

In [1]:
import pandas as pd

ice = pd.read_csv("../downloads/ice/ice_cot.csv", low_memory=False)
marouen = pd.read_csv("../cot_data/cot_db.csv")

## Step 1 — Find Gasoil in the ICE file

In [2]:
ice["market"] = ice["Market_and_Exchange_Names"].fillna(
    ice.get("\ufeffMarket_and_Exchange_Names", "")
)

# What Gasoil markets exist?
gasoil = ice[ice["market"].str.contains("Gasoil", case=False, na=False)]
print("Gasoil markets in ICE file:")
for name, count in gasoil["market"].value_counts().items():
    print(f"  {count:>4d} rows  {name}")

# What columns do we use?
print("\nColumns used for Commercial:")
print("  Prod_Merc_Positions_Long_All")
print("  Prod_Merc_Positions_Short_All")
print("\nColumns used for ManagedMoney (= MM + Other):")
print("  M_Money_Positions_Long_All + Other_Rept_Positions_Long_All")
print("  M_Money_Positions_Short_All + Other_Rept_Positions_Short_All")

Gasoil markets in ICE file:
   909 rows  ICE Gasoil Futures - ICE Futures Europe
   681 rows  ICE Gasoil Futures and Options - ICE Futures Europe

Columns used for Commercial:
  Prod_Merc_Positions_Long_All
  Prod_Merc_Positions_Short_All

Columns used for ManagedMoney (= MM + Other):
  M_Money_Positions_Long_All + Other_Rept_Positions_Long_All
  M_Money_Positions_Short_All + Other_Rept_Positions_Short_All


## Step 2 — Filter for Combined, fall back to FutOnly pre-2013

In [3]:
gasoil_comb = ice[ice["market"].str.contains(r"Gasoil.*Options", case=False, na=False)].copy()
gasoil_fo = ice[ice["market"].str.contains(r"Gasoil.*Futures\b", case=False, na=False)].copy()
gasoil_fo = gasoil_fo[~gasoil_fo["market"].str.contains("Options", case=False, na=False)]

for df in [gasoil_comb, gasoil_fo]:
    df["date"] = pd.to_datetime(df["As_of_Date_Form_MM/DD/YYYY"], format="mixed")
    df.drop_duplicates(subset=["date"], keep="first", inplace=True)

print(f"Combined:  {len(gasoil_comb)} rows, {gasoil_comb['date'].min().date()} to {gasoil_comb['date'].max().date()}")
print(f"FutOnly:   {len(gasoil_fo)} rows, {gasoil_fo['date'].min().date()} to {gasoil_fo['date'].max().date()}")

Combined:  681 rows, 2013-03-12 to 2026-03-24
FutOnly:   795 rows, 2011-01-04 to 2026-03-24


In [4]:
fo_only = gasoil_fo[~gasoil_fo["date"].isin(gasoil_comb["date"])]
gasoil_all = pd.concat([gasoil_comb, fo_only], ignore_index=True).sort_values("date")

print(f"Combined + FutOnly fallback: {len(gasoil_all)} rows")
print(f"  of which {len(fo_only)} are FutOnly fallback (pre-{gasoil_comb['date'].min().date()})")

Combined + FutOnly fallback: 795 rows
  of which 114 are FutOnly fallback (pre-2013-03-12)


## Step 3 — Extract Commercial and ManagedMoney

In [5]:
cols = [
    "Prod_Merc_Positions_Long_All", "Prod_Merc_Positions_Short_All",
    "M_Money_Positions_Long_All", "M_Money_Positions_Short_All",
    "Other_Rept_Positions_Long_All", "Other_Rept_Positions_Short_All",
]
for c in cols:
    gasoil_all[c] = pd.to_numeric(gasoil_all[c], errors="coerce")

rebuilt = pd.DataFrame({
    "date": gasoil_all["date"].values,
    "CommercialLong":  gasoil_all["Prod_Merc_Positions_Long_All"].values,
    "CommercialShort": gasoil_all["Prod_Merc_Positions_Short_All"].values,
    "MMLong":  gasoil_all["M_Money_Positions_Long_All"].values + gasoil_all["Other_Rept_Positions_Long_All"].values,
    "MMShort": gasoil_all["M_Money_Positions_Short_All"].values + gasoil_all["Other_Rept_Positions_Short_All"].values,
})
rebuilt["CommercialNet"] = rebuilt["CommercialLong"] - rebuilt["CommercialShort"]
rebuilt["MMNet"] = rebuilt["MMLong"] - rebuilt["MMShort"]

print(f"Rebuilt QS: {len(rebuilt)} rows")
rebuilt.head()

Rebuilt QS: 795 rows


,date,CommercialLong,CommercialShort,MMLong,MMShort,CommercialNet,MMNet
0,2011-01-04,241271,464782,58146,6955,-223511,51191
1,2011-01-11,222508,449078,60433,5897,-226570,54536
2,2011-01-18,240277,476526,70577,4494,-236249,66083
3,2011-01-25,260507,498316,73638,4897,-237809,68741
4,2011-02-01,279903,517730,76790,5894,-237827,70896


## Step 4 — Compare against Marouen's cot_db

In [6]:
marouen["tradeDate"] = pd.to_datetime(marouen["tradeDate"])
qs = marouen[marouen["Name"] == "QS"].dropna(subset=["Commercial_NetPosition"]).copy()

comp = pd.merge(rebuilt, qs, left_on="date", right_on="tradeDate", how="inner")

cutoff = gasoil_comb["date"].min()
print(f"Matched rows: {len(comp)} (Marouen has {len(qs)})")
print(f"Combined data starts: {cutoff.date()}\n")

pairs = [
    ("CommercialLong",  "CommercialLongPosition"),
    ("CommercialShort", "CommercialShortPosition"),
    ("CommercialNet",   "Commercial_NetPosition"),
    ("MMLong",          "ManagedMoney_LongPosition"),
    ("MMShort",         "ManagedMoney_ShortPosition"),
    ("MMNet",           "ManagedMoney_NetPosition"),
]

for label, mask in [("Combined (2013+)",            comp["date"] >= cutoff),
                     ("FutOnly fallback (pre-2013)", comp["date"] < cutoff)]:
    sub = comp[mask]
    if sub.empty:
        continue
    print(f"--- {label}: {len(sub)} rows ---")
    print(f"{'rebuilt':20s}   {'marouen':30s}   {'MAPE':>8s}   {'MAE':>8s}   {'max_diff':>8s}")
    print("-" * 90)
    for ours, theirs in pairs:
        abs_diff = (sub[ours] - sub[theirs]).abs()
        mae = abs_diff.mean()
        denom = sub[theirs].abs().replace(0, float("nan"))
        mape = (abs_diff / denom * 100).mean()
        print(f"{ours:20s}   {theirs:30s}   {mape:7.4f}%   {mae:8.2f}   {abs_diff.max():.0f}")
    print()

Matched rows: 753 (Marouen has 757)
Combined data starts: 2013-03-12

--- Combined (2013+): 641 rows ---
rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition            0.0000%       0.00   0
CommercialShort        CommercialShortPosition           0.0000%       0.00   0
CommercialNet          Commercial_NetPosition            0.0000%       0.00   0
MMLong                 ManagedMoney_LongPosition         0.0000%       0.00   0
MMShort                ManagedMoney_ShortPosition        0.0000%       0.00   0
MMNet                  ManagedMoney_NetPosition          0.0000%       0.00   0

--- FutOnly fallback (pre-2013): 112 rows ---
rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLo